# Benchmarking model inference on 256-channel segments

This notebook loads the independently trained models and saves raw timing
samples, summary statistics, and the timing protocol. It does not train
a model and does not make a figure.

The timed scope is deliberately limited to the trained ML pipeline:
CNN forward pass, sigmoid, and thresholding; or MLP pipeline scoring and
thresholding. File reading, feature calculation, maximum normalization,
and CPU-to-GPU transfer are prepared outside the timer. This is the
model-only comparison requested for the report.


In [ ]:
from __future__ import annotations

import json
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd


WORK_ROOT = Path.cwd().resolve()

DATA_FOLDER = Path("/hercules/results/akazantsev/rfim_dataset")
META_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_channels_meta.csv"
SPLIT_PATH = DATA_FOLDER / "split_indices.npz"
PROFILES_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_channels.npy"

# The training subset deliberately retains only statistical features and labels.
# These full files retain channel and segment identity and are used only by the
# inference-timing notebook, where one input must correspond to a real 256-channel
# observation segment.
FULL_META_PATH = DATA_FOLDER / "B0531+21_59000_48386_channels_meta.csv"
FULL_PROFILES_PATH = DATA_FOLDER / "B0531+21_59000_48386_channels.npy"
SUBSET_SOURCE_INDICES_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_indices.npy"

# Change the tag only for a deliberate new experiment. Existing results are never overwritten.
RUN_TAG = "b0531_legacy_performance_v1"
RUN_ROOT = WORK_ROOT / "outputs" / "performance_comparison" / RUN_TAG


def json_ready(value):
    if isinstance(value, dict):
        return {key: json_ready(item) for key, item in value.items()}
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value


def write_json(path: Path, payload: dict) -> None:
    with path.open("w", encoding="utf-8") as handle:
        json.dump(json_ready(payload), handle, indent=2, sort_keys=True)
        handle.write("\n")


def git_revision() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=WORK_ROOT,
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None


In [ ]:
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
class CNN1DRFI256Logits(nn.Module):
    """Legacy 1D-CNN architecture for one 256-sample channel profile."""

    def __init__(self, dropout: float = 0.5):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=7)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=10)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(256 * 25, 256)
        self.fc2 = nn.Linear(256, 1)

    def forward(self, x):
        x = F.max_pool1d(F.relu(self.conv1(x)), kernel_size=2)
        # The legacy architecture pads only this intermediate activation map.
        x = F.pad(x, (0, 1))
        x = F.max_pool1d(F.relu(self.conv2(x)), kernel_size=2)
        x = F.max_pool1d(F.relu(self.conv3(x)), kernel_size=2)
        x = x.flatten(start_dim=1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(-1)


In [ ]:
NSAMP = 256
TSAMP_US = 102.4
BUDGET_MS = NSAMP * TSAMP_US * 1e-3
WARMUP_CALLS = 20
SEGMENTS_PER_ESTIMATE = [100, 1000]
RANDOM_STATE = 42

# Existing training artifacts are immutable inputs. Benchmark outputs
# live separately, so rerunning a measurement cannot be confused with
# retraining a model.
results_root = WORK_ROOT / "outputs" / "performance_comparison"
training_root = results_root / "b0531_legacy_performance_v1"
benchmark_dir = results_root / "benchmark_runs" / "b0531_full_segments_model_only"

cnn_cpu_dir = training_root / "cnn_cpu_legacy_max"
cnn_gpu_dir = training_root / "cnn_cuda_legacy_max"
mlp_dir = training_root / "mlp_orig_top3"
output_dir = benchmark_dir

required = [
    cnn_cpu_dir / "checkpoint.pt",
    cnn_gpu_dir / "checkpoint.pt",
    mlp_dir / "mlp_orig_top3.joblib",
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Run the required training notebooks first:\n" + "\n".join(map(str, missing)))
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required because this benchmark includes the CNN GPU run.")
if output_dir.exists():
    raise FileExistsError(
        f"{output_dir} already exists. Create a new explicitly named benchmark directory rather than overwrite it."
    )
output_dir.mkdir(parents=True)

print(f"Real-time budget: {BUDGET_MS:.4f} ms per {NSAMP}-sample segment")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# The subset used for training deliberately contains only feature columns
# and labels. It cannot be reassembled into physical 256-channel segments.
# Frozen-model timing therefore uses the full row-per-channel data, where
# segment_index and channel_index preserve the actual input geometry.
required_full_files = [FULL_META_PATH, FULL_PROFILES_PATH]
missing_full_files = [path for path in required_full_files if not path.exists()]
if missing_full_files:
    raise FileNotFoundError(
        "Segment-level timing requires the full row-per-channel dataset. "
        "The training subset does not retain segment membership. Missing:\n"
        + "\n".join(map(str, missing_full_files))
    )

meta = pd.read_csv(FULL_META_PATH).fillna("None")
profiles = np.load(FULL_PROFILES_PATH, mmap_mode="r")

if len(meta) != len(profiles):
    raise ValueError("Full metadata and profile array have different row counts.")
required_segment_columns = {"segment_index", "channel_index"}
missing_segment_columns = required_segment_columns.difference(meta.columns)
if missing_segment_columns:
    raise ValueError(
        "The full metadata is missing the columns required for physical "
        f"segment timing: {sorted(missing_segment_columns)}"
    )

selected_features = ["mean_o", "std_o", "skew_o"]
missing_features = set(selected_features).difference(meta.columns)
if missing_features:
    raise ValueError(f"Metadata is missing selected MLP features: {sorted(missing_features)}")

grouped_indices = []
for _, group in meta.groupby("segment_index", sort=True):
    if len(group) != NSAMP or group["channel_index"].nunique() != NSAMP:
        continue
    grouped_indices.append(
        group.sort_values("channel_index").index.to_numpy(dtype=int)
    )
if not grouped_indices:
    raise ValueError(
        "No complete 256-channel physical segment is available in the full dataset."
    )

print(f"Complete physical segments available: {len(grouped_indices)}")


In [ ]:
def normalize_legacy_max(segment_rows: np.ndarray) -> np.ndarray:
    values = np.asarray(segment_rows, dtype=np.float32)
    maximum = values.max(axis=1, keepdims=True)
    maximum = np.where(maximum < 1e-8, 1.0, maximum)
    return values / maximum


def load_cnn(device_name: str, checkpoint_path: Path):
    device = torch.device(device_name)
    model = CNN1DRFI256Logits(dropout=0.5).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, device


cnn_cpu, cpu_device = load_cnn("cpu", cnn_cpu_dir / "checkpoint.pt")
cnn_gpu, gpu_device = load_cnn("cuda", cnn_gpu_dir / "checkpoint.pt")
mlp_bundle = joblib.load(mlp_dir / "mlp_orig_top3.joblib")
mlp_pipeline = mlp_bundle["pipeline"]
mlp_threshold = float(mlp_bundle["threshold"])

if mlp_bundle["feature_cols"] != selected_features:
    raise ValueError("The loaded MLP does not use the declared selected feature order.")


def choose_segments(n_segments: int, rng: np.random.Generator):
    positions = rng.choice(
        len(grouped_indices), size=n_segments, replace=n_segments > len(grouped_indices)
    )
    return [grouped_indices[int(position)] for position in positions]


def prepared_cnn_input(row_indices: np.ndarray, device: torch.device):
    values = normalize_legacy_max(profiles[row_indices])
    return torch.from_numpy(values[:, None, :]).to(device)


def prepared_mlp_input(row_indices: np.ndarray):
    return meta.iloc[row_indices][selected_features].apply(
        pd.to_numeric, errors="coerce"
    )


def synchronize(device: torch.device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


@torch.no_grad()
def benchmark_cnn(model, device, segments, label):
    prepared = [prepared_cnn_input(indices, device) for indices in segments]
    for tensor in prepared[:WARMUP_CALLS]:
        _ = torch.sigmoid(model(tensor)) >= 0.5
    synchronize(device)

    rows = []
    for order, tensor in enumerate(prepared):
        synchronize(device)
        started = time.perf_counter()
        _ = torch.sigmoid(model(tensor)) >= 0.5
        synchronize(device)
        rows.append({"model": label, "segment_order": order, "duration_ms": (time.perf_counter() - started) * 1e3})
    return rows


def benchmark_mlp(segments):
    prepared = [prepared_mlp_input(indices) for indices in segments]
    for features in prepared[:WARMUP_CALLS]:
        _ = mlp_pipeline.predict_proba(features)[:, 1] >= mlp_threshold

    rows = []
    for order, features in enumerate(prepared):
        started = time.perf_counter()
        _ = mlp_pipeline.predict_proba(features)[:, 1] >= mlp_threshold
        rows.append({"model": "MLP (3 features)", "segment_order": order, "duration_ms": (time.perf_counter() - started) * 1e3})
    return rows


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
timing_rows = []

for n_segments in SEGMENTS_PER_ESTIMATE:
    segment_indices = choose_segments(n_segments, rng)
    benchmark_runs = [
        ("MLP (3 features)", "cpu", benchmark_mlp(segment_indices)),
        ("1D CNN", "cpu", benchmark_cnn(cnn_cpu, cpu_device, segment_indices, "1D CNN")),
        ("1D CNN", "cuda", benchmark_cnn(cnn_gpu, gpu_device, segment_indices, "1D CNN")),
    ]
    for model, device_name, rows in benchmark_runs:
        for row in rows:
            row.update({
                "device": device_name,
                "n_segments_in_estimate": n_segments,
                "budget_ms": BUDGET_MS,
                "timing_scope": "model_only",
            })
            timing_rows.append(row)

samples = pd.DataFrame(timing_rows)
samples.to_csv(output_dir / "inference_timing_samples.csv", index=False)

summary = (
    samples.groupby(["model", "device", "n_segments_in_estimate", "budget_ms"], as_index=False)
    .agg(
        n_measurements=("duration_ms", "size"),
        median_ms=("duration_ms", "median"),
        mean_ms=("duration_ms", "mean"),
        p95_ms=("duration_ms", lambda x: np.percentile(x, 95)),
        min_ms=("duration_ms", "min"),
        max_ms=("duration_ms", "max"),
    )
)
summary["meets_budget_at_p95"] = summary["p95_ms"] <= summary["budget_ms"]
summary["budget_factor_at_p95"] = summary["budget_ms"] / summary["p95_ms"]
summary.to_csv(output_dir / "inference_timing_summary.csv", index=False)

def find_training_summary(run_dir: Path, device_name: str) -> Path:
    candidates = [
        run_dir / "training_summary.json",
        run_dir / f"training_summary_{device_name}.json",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate a training summary. Looked for:\n"
        + "\n".join(map(str, candidates))
    )


training_rows = []
for label, device_name, run_dir in [
    ("MLP (3 features)", "cpu", mlp_dir),
    ("1D CNN", "cpu", cnn_cpu_dir),
    ("1D CNN", "cuda", cnn_gpu_dir),
]:
    summary_path = find_training_summary(run_dir, device_name)
    with summary_path.open(encoding="utf-8") as handle:
        run_summary = json.load(handle)
    training_rows.append({
        "model": label,
        "device": device_name,
        "training_wall_clock_s": run_summary["training_wall_clock_s"],
        "training_timer_scope": run_summary["training_protocol"]["timer_scope"],
    })
pd.DataFrame(training_rows).to_csv(output_dir / "training_time_summary.csv", index=False)

protocol = {
    "training_root": training_root,
    "benchmark_dir": output_dir,
    "budget_ms": BUDGET_MS,
    "segment_definition": "one complete 256-channel by 256-sample physical observation segment",
    "input_data": {
        "metadata_path": FULL_META_PATH,
        "profiles_path": FULL_PROFILES_PATH,
        "selection": "deterministic random sample from complete full-data segments; input content does not affect model-only timing",
    },
    "models": ["MLP (3 features)", "1D CNN CPU", "1D CNN GPU"],
    "timing_scope": "trained model and decision only; excludes file loading, feature calculation, legacy maximum normalization, and host-to-device transfer",
    "warmup_calls": WARMUP_CALLS,
    "segments_per_estimate": SEGMENTS_PER_ESTIMATE,
    "random_state": RANDOM_STATE,
    "gpu_name": torch.cuda.get_device_name(0),
    "torch_version": torch.__version__,
    "python_version": platform.python_version(),
    "code_revision": git_revision(),
}
write_json(output_dir / "timing_protocol.json", protocol)

display(summary.sort_values(["n_segments_in_estimate", "model", "device"]))
